In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
df['prompt_len']=df['prompt'].str.len()
for col in ['A','B','C','D','E']:
    df[f'{col}_len']=df[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_df=df.apply(get_overlaps,axis=1)
df=pd.concat([df,overlaps_df],axis=1)

In [10]:
from sklearn.model_selection import train_test_split

x_train,x_val,y_train,y_val=train_test_split(df,answer,train_size=0.8,random_state=1)

In [11]:
type(x_train)

pandas.core.frame.DataFrame

In [12]:
answer_prior=y_train.value_counts(normalize=True).to_dict()
prior_list=[answer_prior[c] for c in ['A','B','C','D','E']]

position_features=np.tile(prior_list,(len(x_train),1))
position_df_train = pd.DataFrame(position_features,
    columns=[f'{c}_prior' for c in ['A','B','C','D','E']], index=x_train.index)
x_train = pd.concat([x_train, position_df_train], axis=1)
position_features_val = np.tile(prior_list, (len(x_val), 1))
position_df_val = pd.DataFrame(position_features_val,
    columns=[f'{c}_prior' for c in ['A','B','C','D','E']], index=x_val.index)
x_val = pd.concat([x_val, position_df_val], axis=1)

In [13]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [14]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [15]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1

In [16]:
!pip install wandb -q

In [17]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [18]:
def len_scores(row):
    lengths=[row['A_len'],row['B_len'],row['C_len'],row['D_len'],row['E_len']]
    max_len=max(lengths)
    min_len=min(lengths)
    scores=[]
    for length in lengths:
        score=0
        if length==max_len:
            score=0.5
        elif length==min_len:
            score=0.2
        else:
            score=0.3
        scores.append(score)
    return scores

length_train_scores=x_train.apply(len_scores,axis=1).to_list()
length_labels=score_to_labels(length_train_scores)
length_map3=map_at_3(length_labels,y_train)
print(length_map3)

0.5320833333333334


In [19]:
length_val_scores=x_val.apply(len_scores,axis=1).to_list()
length_val_labels=score_to_labels(length_val_scores)
length_val_map3=map_at_3(length_val_labels,y_val)
length_val_map3

0.49666666666666665

In [20]:
from sklearn.metrics import accuracy_score,f1_score

len_train_acc,len_val_acc,len_train_f1,len_val_f1=support_metrics(length_train_scores,length_val_scores)

In [21]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="largestlen-baseline-metrics",id="largelen_v1",resume="allow")
    wandb.log({
        "map_at_3_train":length_map3,   
        "map_at_3_val":length_val_map3,
        "accuracy_at_train":len_train_acc,
        "accuracy_at_val":len_val_acc,
        "f1_at_train":len_train_f1,
        "f1_at_val":len_val_f1,
        "metric":"Longest-Length"
        })
    wandb.finish()

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260722_071024-largelen_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run largestlen-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/largelen_v1
wandb: updating run metadata
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.405
wandb:   accuracy_at_val 0.3825
wandb:       f1_at_train 0.40268
wandb:         f1_at_val 0.37871
wandb:    map_at_3_train 0.53208
wandb:   

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True,ngram_range=(1,2))
all_text=x_train['prompt'].tolist()+x_train['A'].tolist()+x_train['B'].tolist()+x_train['C'].tolist()+x_train['D'].tolist()+x_train['E'].tolist()
tfidf.fit(all_text)

TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

In [23]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [24]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [25]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

0.5355208333333333
0.5320833333333334


In [26]:
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [27]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics",id="tfidf_v1",resume="allow")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

wandb: setting up run tfidf_v1
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260722_071028-tfidf_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf_v1
wandb: updating run metadata
wandb: uploading history steps 86-86, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.34813
wandb:   accuracy_at_val 0.3525
wandb:       f1_at_train 0.34639
wandb:         f1_at_val 0.35202
wandb:    map_at_3_train 0.53552
wandb:      map_at_3_val 0.53208
wandb:      sublinear_tf T

In [28]:
label_map={"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

y_train_idx=y_train.map(label_map).values if y_train.dtype=='object' else y_train.values
y_val_idx=y_val.map(label_map).values if y_val.dtype=='object' else y_val.values

In [29]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [30]:
option_cols=['A','B','C','D','E']
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [31]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [32]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [33]:
import torch
import numpy as np
import random

seed=42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [34]:
from torch.utils.data import TensorDataset,DataLoader

x_train_t=torch.tensor(np.concatenate((tfidf_train_scores,np.array(length_train_scores),w2v_train_scores,x_train[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap','A_prior','B_prior','C_prior','D_prior','E_prior']]),axis=1),dtype=torch.float32)
x_val_t=torch.tensor(np.concatenate((tfidf_val_scores,np.array(length_val_scores),w2v_val_scores,x_val[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap','A_prior','B_prior','C_prior','D_prior','E_prior']]),axis=1),dtype=torch.float32)
y_train_t=torch.tensor(y_train_idx,dtype=torch.long)
y_val_t=torch.tensor(y_val_idx,dtype=torch.long)

train_ds=TensorDataset(x_train_t,y_train_t)
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True)

In [35]:
import torch.nn as nn

class MCQANN(nn.Module):
    def __init__(self,input_dim=25,num_choices=5,hidden_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim,num_choices)
        )
    def forward(self,x):
        return self.net(x)

model=MCQANN(input_dim=25)

In [36]:
wandb.init(project="24f2002314-t22026",name="MCQ-ANN-v4",id="ann_version4",resume="allow")
wandb.log({
        "lr":2e-3,   
        "weight_decay":1e-4,
        "epochs": 500,
        "patience": 15,
        "batch_size": 64,
        "architecture": "MCQANN",
        "input_dim": x_train_t.shape[1],})

wandb: setting up run ann_version4
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260722_071059-ann_version4
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run MCQ-ANN-v4
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/ann_version4


In [37]:
criteria=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=2e-3,weight_decay=1e-4)
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

best_val_loss=float('inf')
patience=15
patience_count=0
best_val_map3=0
EPOCHS=500
for epoch in range(EPOCHS):
    model.train()
    total_loss=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output=model(x)
        loss=criteria(output,y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    if epoch%1==0:
        model.eval()
        with torch.no_grad():
            val_ds=x_val_t.to(device)
            val_out=model(val_ds)
            val_loss=criteria(val_out,y_val_t.to(device))
            val_acc=(val_out.argmax(dim=1).cpu()==y_val_t).float().mean()
            val_scores=val_out.cpu().numpy()
            val_labels=score_to_labels(val_scores,top1=False)
            val_map3=map_at_3(val_labels, y_val)
            avg_train_loss=total_loss/len(train_loader)
    if epoch%5==0:
        print(f'Epoch :{epoch} - train loss : {avg_train_loss:.4f} val loss : {val_loss:.4f} val_acc={val_acc:.4f} val_map3={val_map3:.4f}')
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": val_loss.item(),
            "val_acc": val_acc.item(),
            "val_map3": val_map3
        })
            
    if val_map3>best_val_map3:
        best_val_loss=val_loss
        best_val_map3=val_map3
        patience_count=0
        best_state=model.state_dict()  
    else:
        patience_count+=1
        if patience_count>=patience:
            print(f"Early stopping at epoch {epoch}")
            break
model.load_state_dict(best_state)

Epoch :0 - train loss : 1.6175 val loss : 1.5908 val_acc=0.2425 val_map3=0.4192
Epoch :5 - train loss : 1.5227 val loss : 1.5169 val_acc=0.3075 val_map3=0.4883
Epoch :10 - train loss : 1.3347 val loss : 1.3001 val_acc=0.4325 val_map3=0.6217
Epoch :15 - train loss : 1.2008 val loss : 1.1470 val_acc=0.5700 val_map3=0.7083
Epoch :20 - train loss : 1.1240 val loss : 1.0906 val_acc=0.5700 val_map3=0.7163
Epoch :25 - train loss : 1.0091 val loss : 1.0328 val_acc=0.5800 val_map3=0.7288
Epoch :30 - train loss : 0.9686 val loss : 0.9577 val_acc=0.6300 val_map3=0.7496
Epoch :35 - train loss : 0.9271 val loss : 0.9360 val_acc=0.6100 val_map3=0.7488
Epoch :40 - train loss : 0.9122 val loss : 0.8426 val_acc=0.6900 val_map3=0.8000
Epoch :45 - train loss : 0.9010 val loss : 0.8281 val_acc=0.7250 val_map3=0.8279
Epoch :50 - train loss : 0.8373 val loss : 0.7531 val_acc=0.7425 val_map3=0.8371
Epoch :55 - train loss : 0.7894 val loss : 0.6943 val_acc=0.7725 val_map3=0.8538
Epoch :60 - train loss : 0.739

<All keys matched successfully>

In [38]:
wandb.run.summary["best_val_loss"]=best_val_loss.item()
wandb.run.summary["best_val_map3"]=val_map3   
wandb.run.summary["stopped_epoch"]=epoch
torch.save(best_state,"best_model.pt")
artifact=wandb.Artifact("mcq_ann_best_model",type="model")
artifact.add_file("best_model.pt")
wandb.log_artifact(artifact)
wandb.finish()

wandb: uploading history steps 17-33, summary, console lines 0-16; uploading artifact mcq_ann_best_model; updating run metadata
wandb: uploading config.yaml; uploading output.log; uploading wandb-summary.json
wandb: uploading summary
wandb: 
wandb: Run history:
wandb: batch_size ▁
wandb:      epoch ▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
wandb:     epochs ▁
wandb:  input_dim ▁
wandb:         lr ▁
wandb:   patience ▁
wandb: train_loss █▇▆▅▄▃▃▃▃▂▂▂▁▁▁▁
wandb:    val_acc ▁▂▃▅▅▅▆▆▇▇██████
wandb:   val_loss █▇▆▅▄▄▃▃▃▂▂▂▁▁▂▁
wandb:   val_map3 ▁▂▄▆▆▆▆▆▇███████
wandb:         +1 ...
wandb: 
wandb: Run summary:
wandb:  architecture MCQANN
wandb:    batch_size 64
wandb: best_val_loss 0.66314
wandb: best_val_map3 0.87625
wandb:         epoch 75
wandb:        epochs 500
wandb:     input_dim 25
wandb:            lr 0.002
wandb:      patience 15
wandb: stopped_epoch 76
wandb:            +5 ...
wandb: 
wandb: 🚀 View run MCQ-ANN-v4 at: https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/

In [39]:
"""bert_train_acc,bert_val_acc,bert_train_f1,bert_val_f1=support_metrics(bert_train_scores,bert_val_scores)
roberta_train_acc,roberta_val_acc,roberta_train_f1,roberta_val_f1=support_metrics(roberta_train_scores,roberta_val_scores)
st_train_acc,st_val_acc,st_train_f1,st_val_f1=support_metrics(st_train_scores,st_val_scores)"""

'bert_train_acc,bert_val_acc,bert_train_f1,bert_val_f1=support_metrics(bert_train_scores,bert_val_scores)\nroberta_train_acc,roberta_val_acc,roberta_train_f1,roberta_val_f1=support_metrics(roberta_train_scores,roberta_val_scores)\nst_train_acc,st_val_acc,st_train_f1,st_val_f1=support_metrics(st_train_scores,st_val_scores)'

In [40]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="BertEmbedding-metrics",id="bert_v1", resume="allow")
    wandb.log({
        "map_at_3_train":bert_train,   
        "map_at_3_val":bert_val,
        "accuracy_at_train":bert_train_acc,
        "accuracy_at_val":bert_val_acc,
        "f1_at_train":bert_train_f1,
        "f1_at_val":bert_val_f1,
        "embedding_model":"BERT",
        "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="BertEmbedding-metrics",id="bert_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":bert_train,   \n        "map_at_3_val":bert_val,\n        "accuracy_at_train":bert_train_acc,\n        "accuracy_at_val":bert_val_acc,\n        "f1_at_train":bert_train_f1,\n        "f1_at_val":bert_val_f1,\n        "embedding_model":"BERT",\n        "max_length":128})\n    wandb.finish()'

In [41]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="RobertaEmbedding-metrics",id="roberta_v1", resume="allow")
    wandb.log({
        "map_at_3_train":roberta_train,   
        "map_at_3_val":roberta_val,
        "accuracy_at_train":roberta_train_acc,
        "accuracy_at_val":roberta_val_acc,
        "f1_at_train":roberta_train_f1,
        "f1_at_val":roberta_val_f1,
        "embedding_model":"ROBERTA",
         "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="RobertaEmbedding-metrics",id="roberta_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":roberta_train,   \n        "map_at_3_val":roberta_val,\n        "accuracy_at_train":roberta_train_acc,\n        "accuracy_at_val":roberta_val_acc,\n        "f1_at_train":roberta_train_f1,\n        "f1_at_val":roberta_val_f1,\n        "embedding_model":"ROBERTA",\n         "max_length":128})\n    wandb.finish()'

In [42]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="SentenceTransformer-metrics",id="st_v1", resume="allow")
    wandb.log({
        "map_at_3_train":st_train,   
        "map_at_3_val":st_val,
        "accuracy_at_train":st_train_acc,
        "accuracy_at_val":st_val_acc,
        "f1_at_train":st_train_f1,
        "f1_at_val":st_val_f1,
        "embedding_model":"Sentence Transformer",
        "max_length":128})
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="SentenceTransformer-metrics",id="st_v1", resume="allow")\n    wandb.log({\n        "map_at_3_train":st_train,   \n        "map_at_3_val":st_val,\n        "accuracy_at_train":st_train_acc,\n        "accuracy_at_val":st_val_acc,\n        "f1_at_train":st_train_f1,\n        "f1_at_val":st_val_f1,\n        "embedding_model":"Sentence Transformer",\n        "max_length":128})\n    wandb.finish()'

In [43]:
"""from transformers import pipeline

classifier1=pipeline("zero-shot-classification",model="microsoft/deberta-base-mnli",device=0)

def zero_shot_apply(row,classifier,labels=True):
    options=[row[c] for c in option_cols]
    result=classifier(row['prompt'],candidate_labels=options)
    ranked=result['labels'][:3]
    scores=result['scores']
    label_map={row[c]:c for c in option_cols}
    if labels:
        return " ".join([label_map[opt] for opt in ranked]),scores

z1_train_results=x_train.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_train_labels=[r[0] for r in z1_train_results]
z1_train_scores=[r[1] for r in z1_train_results]

z1_val_results=x_val.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_val_labels=[r[0] for r in z1_val_results]
z1_val_scores=[r[1] for r in z1_val_results]

z1_train_map3=map_at_3(z1_train_labels,y_train)
z1_val_map3=map_at_3(z1_val_labels,y_val)"""

'from transformers import pipeline\n\nclassifier1=pipeline("zero-shot-classification",model="microsoft/deberta-base-mnli",device=0)\n\ndef zero_shot_apply(row,classifier,labels=True):\n    options=[row[c] for c in option_cols]\n    result=classifier(row[\'prompt\'],candidate_labels=options)\n    ranked=result[\'labels\'][:3]\n    scores=result[\'scores\']\n    label_map={row[c]:c for c in option_cols}\n    if labels:\n        return " ".join([label_map[opt] for opt in ranked]),scores\n\nz1_train_results=x_train.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()\nz1_train_labels=[r[0] for r in z1_train_results]\nz1_train_scores=[r[1] for r in z1_train_results]\n\nz1_val_results=x_val.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()\nz1_val_labels=[r[0] for r in z1_val_results]\nz1_val_scores=[r[1] for r in z1_val_results]\n\nz1_train_map3=map_at_3(z1_train_labels,y_train)\nz1_val_map3=map_at_3(z1_val_labels,y_val)'

In [44]:
from datasets import Dataset

label_map={'A':0,'B':1,'C':2,'D':3,'E':4}

x_tr=x_train.copy()
x_v=x_val.copy()
x_tr['label']=y_train.map(label_map)
x_v['label']=y_val.map(label_map)

train_dataset=Dataset.from_pandas(x_tr.reset_index(drop=True))
val_dataset=Dataset.from_pandas(x_v.reset_index(drop=True))

In [45]:
!pip install -U -q bitsandbytes>=0.46.1 accelerate peft

In [46]:
"""import torch
from transformers import AutoTokenizer,AutoModelForSequenceClassification,BitsAndBytesConfig
from peft import LoraConfig,TaskType,get_peft_model,prepare_model_for_kbit_training
import torch

model_name1 = "Qwen/Qwen2.5-3B-Instruct"
tokeniser1 = AutoTokenizer.from_pretrained(model_name1)
if tokeniser1.pad_token is None:
    tokeniser1.pad_token = tokeniser1.eos_token

bnb_config =BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model1 =AutoModelForSequenceClassification.from_pretrained(
    model_name1, num_labels=5, quantization_config=bnb_config, device_map="auto"
)
model1.config.pad_token_id = tokeniser1.pad_token_id
model1 = prepare_model_for_kbit_training(model1)
model1.enable_input_require_grads()
model1 = get_peft_model(model1, LoraConfig(
    task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
))

def preprocessing(data, tokeniser):
    texts = [f"Question: {p}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nE. {e}\nAnswer:"
             for p,a,b,c,d,e in zip(data['prompt'],data['A'],data['B'],data['C'],data['D'],data['E'])]
    result = tokeniser(texts, truncation=True, max_length=512, padding='max_length')
    if "label" in data:
        result["label"] = data["label"]
    return result
    
processed_train1=train_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=train_dataset.column_names)
processed_val1=val_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=val_dataset.column_names)"""

'import torch\nfrom transformers import AutoTokenizer,AutoModelForSequenceClassification,BitsAndBytesConfig\nfrom peft import LoraConfig,TaskType,get_peft_model,prepare_model_for_kbit_training\nimport torch\n\nmodel_name1 = "Qwen/Qwen2.5-3B-Instruct"\ntokeniser1 = AutoTokenizer.from_pretrained(model_name1)\nif tokeniser1.pad_token is None:\n    tokeniser1.pad_token = tokeniser1.eos_token\n\nbnb_config =BitsAndBytesConfig(\n    load_in_4bit=True, bnb_4bit_quant_type="nf4",\n    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n)\nmodel1 =AutoModelForSequenceClassification.from_pretrained(\n    model_name1, num_labels=5, quantization_config=bnb_config, device_map="auto"\n)\nmodel1.config.pad_token_id = tokeniser1.pad_token_id\nmodel1 = prepare_model_for_kbit_training(model1)\nmodel1.enable_input_require_grads()\nmodel1 = get_peft_model(model1, LoraConfig(\n    task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,lora_dropout=0.05,\n    target_modules=["q_proj","k_proj","v_p

In [47]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    top3=np.argsort(-logits,axis=1)[:,:3]
    ap=[1.0/(list(row).index(l)+1) if l in row else 0.0 for row,l in zip(top3,labels)]
    acc=(np.argmax(logits, axis=1)==labels).mean()
    preds=np.argmax(logits,axis=1)
    f1_macro=f1_score(labels,preds,average="macro")
    return {"map_at_3":np.mean(ap),"accuracy":acc,"f1_macro":f1_macro}

In [48]:
"""del model
torch.cuda.empty_cache()"""

'del model\ntorch.cuda.empty_cache()'

In [49]:
"""from transformers import TrainingArguments, Trainer

processed_train1.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val1.set_format(type="torch",columns=["input_ids","attention_mask","label"])

train_args1=TrainingArguments(output_dir="./qwen-mcq",learning_rate=5e-4,eval_strategy="epoch",gradient_accumulation_steps=4,gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=3,warmup_ratio=0.06,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=True,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=4,per_device_eval_batch_size=4,load_best_model_at_end=True)

trainer1=Trainer(model=model1,args=train_args1,train_dataset=processed_train1,eval_dataset=processed_val1,compute_metrics=compute_metrics)
trainer1.train()"""

'from transformers import TrainingArguments, Trainer\n\nprocessed_train1.set_format(type="torch",columns=["input_ids","attention_mask","label"])\nprocessed_val1.set_format(type="torch",columns=["input_ids","attention_mask","label"])\n\ntrain_args1=TrainingArguments(output_dir="./qwen-mcq",learning_rate=5e-4,eval_strategy="epoch",gradient_accumulation_steps=4,gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=3,warmup_ratio=0.06,remove_unused_columns=False,max_grad_norm=1.0,\n                             fp16=True,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=4,per_device_eval_batch_size=4,load_best_model_at_end=True)\n\ntrainer1=Trainer(model=model1,args=train_args1,train_dataset=processed_train1,eval_dataset=processed_val1,compute_metrics=compute_metrics)\ntrainer1.train()'

In [50]:
"""val_results=trainer1.predict(processed_val1)
val_logits1=val_results.predictions
train_results=trainer1.predict(processed_train1)
train_logits1=train_results.predictions"""

'val_results=trainer1.predict(processed_val1)\nval_logits1=val_results.predictions\ntrain_results=trainer1.predict(processed_train1)\ntrain_logits1=train_results.predictions'

In [51]:
"""val_results1=trainer1.predict(processed_val1)
val_results2=trainer2.predict(processed_val2)

val_logits1=val_results1.predictions
val_logits2=val_results2.predictions

best_score1=0
best_score2=0
best_w=0.5

label_inverse={0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}
val_labels=[label_inverse[int(l)] for l in list(processed_val1['label'])]
val_labels_idx=[ord(l)-65 for l in val_labels]

for w in np.arange(0.1,1.0,0.1):
    val_blend=w*val_logits1 +(1-w)*val_logits2
    val_blend_labels=score_to_labels(val_blend)
    score1=map_at_3(val_blend_labels,val_labels)
    score2=accuracy_score(np.argmax(val_blend,axis=1),val_labels_idx)
    print(f"w={w:.1f}, MAP@3={score1:.4f}")
    if score1>best_score1:
        best_score1=score1
        best_score2=score2
        best_w=w

print(best_w)
print(best_score1)
print(best_score2)"""

'val_results1=trainer1.predict(processed_val1)\nval_results2=trainer2.predict(processed_val2)\n\nval_logits1=val_results1.predictions\nval_logits2=val_results2.predictions\n\nbest_score1=0\nbest_score2=0\nbest_w=0.5\n\nlabel_inverse={0:\'A\', 1:\'B\', 2:\'C\', 3:\'D\', 4:\'E\'}\nval_labels=[label_inverse[int(l)] for l in list(processed_val1[\'label\'])]\nval_labels_idx=[ord(l)-65 for l in val_labels]\n\nfor w in np.arange(0.1,1.0,0.1):\n    val_blend=w*val_logits1 +(1-w)*val_logits2\n    val_blend_labels=score_to_labels(val_blend)\n    score1=map_at_3(val_blend_labels,val_labels)\n    score2=accuracy_score(np.argmax(val_blend,axis=1),val_labels_idx)\n    print(f"w={w:.1f}, MAP@3={score1:.4f}")\n    if score1>best_score1:\n        best_score1=score1\n        best_score2=score2\n        best_w=w\n\nprint(best_w)\nprint(best_score1)\nprint(best_score2)'

In [52]:
"""if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="Qwen/Qwen2.5-3B-Instructv3",id="Qwen2.5-3B-Instructv3", resume="allow")
    wandb.log({   
        "map_at_3_train": train_results.metrics["test_map_at_3"],
    "map_at_3_val": val_results.metrics["test_map_at_3"],
    "accuracy_at_train": train_results.metrics["test_accuracy"],
    "accuracy_at_val": val_results.metrics["test_accuracy"],
    "f1_at_train": train_results.metrics["test_f1_macro"],
    "f1_at_val": val_results.metrics["test_f1_macro"],
        "MODEL":"Qwen/Qwen2.5-3B-Instruct",
        "epochs":3,"warmup_ratio":0.06,
        "learning_rate":3e-4,
        'gradient_accumulation_steps':4,
         })
    wandb.finish()"""

'if wandb_mode==True:\n    wandb.init(project="24f2002314-t22026",name="Qwen/Qwen2.5-3B-Instructv3",id="Qwen2.5-3B-Instructv3", resume="allow")\n    wandb.log({   \n        "map_at_3_train": train_results.metrics["test_map_at_3"],\n    "map_at_3_val": val_results.metrics["test_map_at_3"],\n    "accuracy_at_train": train_results.metrics["test_accuracy"],\n    "accuracy_at_val": val_results.metrics["test_accuracy"],\n    "f1_at_train": train_results.metrics["test_f1_macro"],\n    "f1_at_val": val_results.metrics["test_f1_macro"],\n        "MODEL":"Qwen/Qwen2.5-3B-Instruct",\n        "epochs":3,"warmup_ratio":0.06,\n        "learning_rate":3e-4,\n        \'gradient_accumulation_steps\':4,\n         })\n    wandb.finish()'

In [53]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test['prompt_len']=test['prompt'].str.len()
for col in ['A','B','C','D','E']:
    test[f'{col}_len']=test[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_test=test.apply(get_overlaps,axis=1)
test=pd.concat([test,overlaps_test],axis=1)

position_features_test = np.tile(prior_list, (len(test), 1))
position_df_test = pd.DataFrame(position_features_test,
    columns=[f'{c}_prior' for c in ['A','B','C','D','E']], index=test.index)
test = pd.concat([test, position_df_test], axis=1)

"""test_dataset=Dataset.from_pandas(test)
processed_test1=test_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=test_dataset.column_names)"""

'test_dataset=Dataset.from_pandas(test)\nprocessed_test1=test_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=test_dataset.column_names)'

In [54]:
"""test_results1=trainer1.predict(processed_test1)
logits1=test_results1.predictions
letters=np.array(["A","B","C","D","E"])
top3=np.argsort(logits1,axis=1)[:, ::-1][:, :3]
top3=[" ".join(letters[row]) for row in top3]
print(top3[:10])"""

'test_results1=trainer1.predict(processed_test1)\nlogits1=test_results1.predictions\nletters=np.array(["A","B","C","D","E"])\ntop3=np.argsort(logits1,axis=1)[:, ::-1][:, :3]\ntop3=[" ".join(letters[row]) for row in top3]\nprint(top3[:10])'

In [55]:
tfidf_test_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(test['prompt']+" "+test[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_test_scores.append(np.asarray(scores).flatten())
tfidf_test_scores=np.array(tfidf_test_scores).T

test_length_scores=test.apply(len_scores,axis=1).to_list()

In [56]:
test_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
w2v_test_scores=[]

for col in option_cols:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(test_prompts,option_matrix))
    w2v_test_scores.append(scores)
w2v_test_scores=np.array(w2v_test_scores).T

In [57]:
X_test=np.concatenate((tfidf_test_scores,np.array(test_length_scores),w2v_test_scores,test[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap','A_prior','B_prior','C_prior','D_prior','E_prior']]),axis=1)
x_test_t=torch.tensor(X_test,dtype=torch.float32)

In [58]:
model.eval()
with torch.no_grad():
    test_out=model(x_test_t.to(device))
test_scores=test_out.cpu().numpy()

top3=score_to_labels(test_scores,top1=False)

In [59]:
submission=pd.DataFrame({'id':test['id'],'Prediction':top3})
submission.to_csv('submission.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      A B D
1   2      B E D
2   3      B D E
3   4      E B D
4   5      E D C
